# Are the GS2 runs converged? R4 scan and R4 hypercube vs R3 / SPR-045 / M1 baselines (Fusion_PhD-38e9.5, port of 6vvq)

Every GS2 run here hit `nstep`, so `exit_reason` says nothing. Convergence is measured from each run's own
gamma(t) through pyrokinetics: `gk_output["growth_rate_tolerance"]` is the RMS of `(gamma(t) - gamma_final)/gamma_final`
over the last 20% of the time axis (`gamma_final` = final point). The spread and drift of gamma and omega over that same window are
recorded alongside.

Classes: **converged** `tol < 1e-2`, **marginal** `1e-2 <= tol < 1e-1`, **unconverged** `tol >= 1e-1`
(a last-point/average agreement of 0.1% needs `tol <= 1e-3`; at `tol >= 1` they are 100-1200% apart).

Port of `results/r4_gs2_convergence/convergence.py` (Slurm array, per-run JSON). Here the per-run
measurement runs in a process pool inside the notebook, so the notebook is the single entry point and nothing is cached on disk.
A relative tolerance on a near-zero or negative gamma is meaningless, so every count is given for all runs and for `gamma > 0` (tail mean).

## Imports and settings

In [ ]:
from multiprocessing import Pool
from pathlib import Path
import os, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from pyrokinetics import Pyro

warnings.filterwarnings("ignore")
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "notebooks").is_dir())
plt.style.use(ROOT / "src/general_analysis/paper.mplstyle")
load_dotenv(ROOT / "local.env", override=True)

analysis_name = "r4_gs2_convergence"
data_root = Path(os.environ["GK_DATA_ROOT"]).expanduser()
code_, run_template = "GS2", "Runs"

# dataset: (project, case, scan_information, run-directory glob). R4 scan and R3 scan are the beta scans; the rest hypercubes.
datasets = {
    "R4_scan":     ("scan_beta", "R4", "parameter_scan_GS2", "ky*"),
    "R3_scan":     ("scan_beta", "R3", "parameter_scan_GS2", "ky*"),
    "R4_lhc":      ("LATIN_HYPERCUBE", "R4", "kbm_8d", "iteration_*"),
    "SPR-045_lhc": ("LATIN_HYPERCUBE", "SPR-045", "kbm_8d", "iteration_*"),
    "M1_lhc":      ("LATIN_HYPERCUBE", "M1", "kbm_8d", "iteration_*"),
}
tail_fraction = 0.2          # last 20% of the time axis
CONV, MARG = 1e-2, 1e-1      # tol < CONV converged; < MARG marginal; else unconverged

## Load and measure

One `Pyro` per run (`load_gk_output()` reads `gs2.out.nc`); failed loads are counted and printed, not dropped silently.

In [ ]:
def measure(d):
    r = dict(dataset=d.parts[-5 if d.parts[-3] != 'scan_beta' else -4], dir=str(d))
    try:
        pyro = Pyro(gk_file=d / "gs2.in", gk_code="GS2")
        pyro.load_gk_output()
        o = pyro.gk_output
        t = o["time"].values
        m = t > (1 - tail_fraction) * t[-1]
        r.update(ky=float(pyro.numerics.ky.m), beta=float(pyro.numerics.beta.m), nt=len(t), T=float(t[-1]), ntail=int(m.sum()),
                 tol=float(np.asarray(o["growth_rate_tolerance"].data.m).ravel()[0]))
        for key, var in (("g", "growth_rate"), ("w", "mode_frequency")):
            y = o[var].values.ravel()[m]
            mean = np.mean(y)
            slope = np.polyfit(t[m], y, 1)[0] if m.sum() > 2 else np.nan
            r.update({key: mean, key + "_spread": np.ptp(y) / abs(mean), key + "_drift": abs(slope) * (t[m][-1] - t[m][0]) / abs(mean)})
    except Exception as e:
        r["error"] = repr(e)[:200]
    return r

dirs = {name: sorted(d for d in (data_root / code_ / run_template / proj / case / scan).glob(g) if (d / "gs2.in").exists())
        for name, (proj, case, scan, g) in datasets.items()}
jobs = [(name, d) for name, ds in dirs.items() for d in ds]
with Pool(int(os.environ.get("SLURM_CPUS_PER_TASK", 4))) as pool:
    recs = pool.map(measure, [d for _, d in jobs], chunksize=8)
df = pd.DataFrame(recs).assign(dataset=[n for n, _ in jobs])
print("runs found", df.groupby("dataset", sort=False).size().to_dict())
print("load failures", df[df["error"].notna()].groupby("dataset").size().to_dict() if "error" in df else "none")

## Counts

In [ ]:
ok = df.dropna(subset=["tol"]).copy()
ok["cls"] = np.where(ok.tol < CONV, "converged", np.where(ok.tol < MARG, "marginal", "unconverged"))
ok["growing"] = ok.g > 0
rows = []
for name in datasets:
    o = ok[ok.dataset == name]
    for lab, sub in (("all", o), ("gamma>0", o[o.growing])):
        rows.append(dict(dataset=name, subset=lab, loaded=len(o), n=len(sub), **{c: int((sub.cls == c).sum()) for c in ("converged", "marginal", "unconverged")},
                         frac_converged=round((sub.cls == "converged").mean(), 2), median_tol=sub.tol.median(), median_g_spread=sub.g_spread.median()))
summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

## Plot: tolerance distribution, R4 tolerance vs ky and beta, worst growing R4 cases

In [ ]:
fig1, ax = plt.subplots(figsize=(7, 4))
bins = np.logspace(-8, 3, 45)
for name in datasets:
    o = ok[ok.dataset == name]
    ax.hist(o.tol.clip(1e-8, 1e3), bins, histtype="step", lw=2 if name.startswith("R4") else 1, label=name)
for x in (CONV, MARG):
    ax.axvline(x, c="k", ls=":")
ax.set(xscale="log")
ax.set_xlabel("tolerance", fontsize=22); ax.set_ylabel("runs", fontsize=22)
ax.tick_params(labelsize=18)
ax.legend(fontsize=22)

fig2, axs = plt.subplots(2, 2, figsize=(10, 7))
for ax, (name, xk) in zip(axs.ravel(), [("R4_scan", "ky"), ("R4_scan", "beta"), ("R4_lhc", "ky"), ("R4_lhc", "beta")]):
    o = ok[ok.dataset == name]
    sc = ax.scatter(o[xk], o.tol.clip(1e-8, 1e3), c=np.log10(o.g.abs().clip(1e-4)), s=6, cmap="viridis")
    ax.axhline(CONV, c="k", ls=":"); ax.axhline(MARG, c="k", ls=":")
    ax.set(yscale="log", xscale="log" if xk == "ky" else "linear")
    ax.set_xlabel(r"$k_y \rho_s$" if xk == "ky" else r"$\beta$", fontsize=22); ax.set_ylabel("tolerance", fontsize=22)
    ax.set_title(name, fontsize=22); ax.tick_params(labelsize=18)
cb = fig2.colorbar(sc, ax=axs); cb.set_label(r"$\log_{10}|\gamma|$", fontsize=22); cb.ax.tick_params(labelsize=18)

worst = pd.concat([ok[(ok.dataset == n) & ok.growing].nlargest(3, "tol") for n in ("R4_scan", "R4_lhc")])
fig3, axs = plt.subplots(2, 3, figsize=(12, 6))
for ax, (_, r) in zip(axs.ravel(), worst.iterrows()):
    o = Pyro(gk_file=Path(r.dir) / "gs2.in", gk_code="GS2")
    o.load_gk_output(); o = o.gk_output
    ax.plot(o["time"].values, o["growth_rate"].values.ravel(), label="gamma")
    ax.plot(o["time"].values, o["mode_frequency"].values.ravel(), label="omega")
    ax.axvline((1 - tail_fraction) * r["T"], c="k", ls=":")
    ax.set_title(r.dataset, fontsize=22)
    ax.set_xlabel("time", fontsize=22); ax.tick_params(labelsize=18)
axs[0, 0].legend(fontsize=22)
plt.show()

## Save

In [ ]:
output_dir = ROOT / "Plots" / analysis_name
output_dir.mkdir(parents=True, exist_ok=True)
fig1.savefig(output_dir / "tolerance_distribution.png", dpi=150, bbox_inches="tight")
fig2.savefig(output_dir / "r4_tolerance_vs_ky_beta.png", dpi=150, bbox_inches="tight")
fig3.savefig(output_dir / "r4_worst_gamma_t.png", dpi=150, bbox_inches="tight")
print("saved", sorted(p.name for p in output_dir.glob("*.png")))

## Interpretation

Left to the reader; findings and the rerun spec are in the Fusion_PhD README for this analysis.